In [23]:
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split

In [24]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-medium")
tokenizer.pad_token = tokenizer.eos_token
tokenizer

GPT2Tokenizer(name_or_path='microsoft/DialoGPT-medium', vocab_size=50257, model_max_length=1024, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
})

In [25]:
model = AutoModelForCausalLM.from_pretrained("microsoft/DialoGPT-medium")
model

Loading weights: 100%|██████████| 293/293 [00:00<00:00, 9905.86it/s]


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 1024)
    (wpe): Embedding(1024, 1024)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-23): 24 x GPT2Block(
        (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=3072, nx=1024)
          (c_proj): Conv1D(nf=1024, nx=1024)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=4096, nx=1024)
          (c_proj): Conv1D(nf=1024, nx=4096)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1024, out_features=50257, bias=False)
)

In [26]:
dataset = pd.read_csv("Mental_Health_FAQ.csv")
dataset.head()

,Question_ID,Questions,Answers
0,1590140,What does it mean to have a mental illness?,Mental illnesses are health conditions that di...
1,2110618,Who does mental illness affect?,It is estimated that mental illness affects 1 ...
2,6361820,What causes mental illness?,It is estimated that mental illness affects 1 ...
3,9434130,What are some of the warning signs of mental i...,Symptoms of mental health disorders vary depen...
4,7657263,Can people with mental illness recover?,"When healing from mental illness, early identi..."


In [27]:
questions = dataset['Questions'].astype('str').values
questions[:5]

array(['What does it mean to have a mental illness?',
       'Who does mental illness affect?', 'What causes mental illness?',
       'What are some of the warning signs of mental illness?',
       'Can people with mental illness recover?'], dtype=object)

In [28]:
questionIds = dataset['Question_ID'].values

In [29]:
answers = dataset['Answers'].astype('str').values
answers[:5]

array(['Mental illnesses are health conditions that disrupt a personâ€™s thoughts, emotions, relationships, and daily functioning. They are associated with distress and diminished capacity to engage in the ordinary activities of daily life.\r\nMental illnesses fall along a continuum of severity: some are fairly mild and only interfere with some aspects of life, such as certain phobias. On the other end of the spectrum lie serious mental illnesses, which result in major functional impairment and interference with daily life. These include such disorders as major depression, schizophrenia, and bipolar disorder, and may require that the person receives care in a hospital.\r\nIt is important to know that mental illnesses are medical conditions that have nothing to do with a personâ€™s character, intelligence, or willpower. Just as diabetes is a disorder of the pancreas, mental illness is a medical condition due to the brainâ€™s biology.\r\nSimilarly to how one would treat diabetes with med

In [30]:
def combineText(example):
    return {
        "text": "User: " + example['question'] + 
            "Bot: " + example['answer']
    }

In [31]:
def encode(example):
    return tokenizer(example['text'], truncation=True, padding=True, max_length=64)

In [32]:
# def add_labels(example):
#     example['labels']=example['input_ids']
#     return example

In [33]:
datasets = Dataset.from_dict({
    "question": questions,
    "answer": answers,
    "labels": questionIds
})
datasets

Dataset({
    features: ['question', 'answer', 'labels'],
    num_rows: 98
})

In [34]:
datasets_split = datasets.train_test_split(test_size=0.2)
datasets_split

DatasetDict({
    train: Dataset({
        features: ['question', 'answer', 'labels'],
        num_rows: 78
    })
    test: Dataset({
        features: ['question', 'answer', 'labels'],
        num_rows: 20
    })
})

In [35]:
trainSet = datasets_split['train']
trainSet

Dataset({
    features: ['question', 'answer', 'labels'],
    num_rows: 78
})

In [36]:
testSet = datasets_split['test']
testSet

Dataset({
    features: ['question', 'answer', 'labels'],
    num_rows: 20
})

In [37]:
trainSet = trainSet.map(combineText)
trainSet

Map: 100%|██████████| 78/78 [00:00<00:00, 972.56 examples/s]


Dataset({
    features: ['question', 'answer', 'labels', 'text'],
    num_rows: 78
})

In [38]:
testSet = testSet.map(combineText)
testSet

Map: 100%|██████████| 20/20 [00:00<00:00, 1249.49 examples/s]


Dataset({
    features: ['question', 'answer', 'labels', 'text'],
    num_rows: 20
})

In [39]:
trainSet = trainSet.map(encode, batched=True)
trainSet

Map: 100%|██████████| 78/78 [00:00<00:00, 1490.98 examples/s]


Dataset({
    features: ['question', 'answer', 'labels', 'text', 'input_ids', 'attention_mask'],
    num_rows: 78
})

In [40]:
testSet = testSet.map(encode, batched=True)
testSet

Map: 100%|██████████| 20/20 [00:00<00:00, 604.19 examples/s]


Dataset({
    features: ['question', 'answer', 'labels', 'text', 'input_ids', 'attention_mask'],
    num_rows: 20
})

In [41]:
# trainSet = trainSet.map(add_labels)
# trainSet

In [42]:
# testSet = testSet.map(add_labels)
# testSet

In [43]:
trainingArgs = TrainingArguments(
    output_dir="./output",
    num_train_epochs=2,
    learning_rate=2e-5,
    per_device_train_batch_size=256,
    per_device_eval_batch_size=256,
    # warmup_steps=500,
    weight_decay=0.01,
    logging_dir=None
)

In [44]:
# trainSet = trainSet.select(range(100))

In [45]:
# testSet = testSet.select(range(670))

In [46]:
trainer=Trainer(
    model=model,
    args=trainingArgs,
    train_dataset= trainSet,
    eval_dataset=testSet
)

In [ ]:
trainer.evaluate(testSet)

c:\Users\donof\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:

trainer.predict(testSet)

c:\Users\donof\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
100%|██████████| 2/2 [00:02<00:00,  1.10s/it]


PredictionOutput(predictions=array([[[ 25.190084,  20.328644,  19.921597, ...,  23.659145,
          22.035862,  24.333487],
        [398.70996 , 360.6712  , 359.0642  , ..., 395.01468 ,
         398.91638 , 410.08542 ],
        [362.1523  , 326.86203 , 320.04373 , ..., 358.0688  ,
         361.63928 , 370.24506 ],
        ...,
        [394.58002 , 353.23868 , 353.7964  , ..., 389.91013 ,
         394.1538  , 413.1651  ],
        [404.5524  , 360.5165  , 361.04745 , ..., 399.4514  ,
         403.95734 , 421.7322  ],
        [413.97247 , 366.0933  , 366.91034 , ..., 404.44147 ,
         410.133   , 429.0277  ]],

       [[ 25.190084,  20.328644,  19.921597, ...,  23.659145,
          22.035862,  24.333487],
        [398.70996 , 360.6712  , 359.0642  , ..., 395.01468 ,
         398.91638 , 410.08542 ],
        [362.1523  , 326.86203 , 320.04373 , ..., 358.0688  ,
         361.63928 , 370.24506 ],
        ...,
        [425.8944  , 382.91476 , 378.88217 , ..., 420.57788 ,
         425.5218

In [ ]:
trainer.train()

100%|██████████| 36/36 [28:57<00:00, 48.27s/it] 

{'train_runtime': 1737.6063, 'train_samples_per_second': 1.266, 'train_steps_per_second': 0.021, 'train_loss': 5.288313971625434, 'epoch': 2.0}


TrainOutput(global_step=36, training_loss=5.288313971625434, metrics={'train_runtime': 1737.6063, 'train_samples_per_second': 1.266, 'train_steps_per_second': 0.021, 'total_flos': 71855308800000.0, 'train_loss': 5.288313971625434, 'epoch': 2.0})

In [ ]:
trainer.save_model("mental-health-dialogpt-FAQ")

In [ ]:
trainer.evaluate(testSet)

c:\Users\donof\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
100%|██████████| 11/11 [01:23<00:00,  7.59s/it]


{'eval_loss': 4.254007339477539,
 'eval_runtime': 91.7075,
 'eval_samples_per_second': 7.306,
 'eval_steps_per_second': 0.12,
 'epoch': 2.0}

In [ ]:
trainer.predict(testSet.select(range(80)))

100%|██████████| 2/2 [00:05<00:00,  2.90s/it]


PredictionOutput(predictions=array([[[ 25.24406 ,  20.362602,  19.935255, ...,  23.59983 ,
          21.976215,  24.345623],
        [309.05075 , 282.32623 , 277.66772 , ..., 309.04352 ,
         312.20865 , 322.577   ],
        [331.07526 , 302.13174 , 296.12918 , ..., 329.902   ,
         336.8476  , 341.6684  ],
        ...,
        [308.21262 , 286.18918 , 281.83606 , ..., 307.9867  ,
         310.70883 , 321.33154 ],
        [341.5949  , 311.69833 , 308.11765 , ..., 336.33142 ,
         340.10712 , 359.07965 ],
        [329.34213 , 296.5484  , 295.2171  , ..., 320.97424 ,
         327.0835  , 337.3223  ]],

       [[ 25.24406 ,  20.362602,  19.935255, ...,  23.59983 ,
          21.976215,  24.345623],
        [309.05075 , 282.32623 , 277.66772 , ..., 309.04352 ,
         312.20865 , 322.577   ],
        [331.07526 , 302.13174 , 296.12918 , ..., 329.902   ,
         336.8476  , 341.6684  ],
        ...,
        [340.55304 , 312.12097 , 306.21292 , ..., 339.97925 ,
         343.5252